In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC_DIR = os.path.join(os.getcwd(), "src")
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from causation_entropy_utils import entropy, mutual_information,\
                                    conditional_entropy,\
                                    causation_entropy

In [36]:
def discretize_quantiles(x, bins=20):
    """
    Discretize a 1D array into approximately equal-frequency bins.
    Returns integer bin labels 0, 1, ..., bins-1.
    """
    edges = np.quantile(x, np.linspace(0, 1, bins + 1))
    # Make edges strictly increasing in case of ties
    edges = np.unique(edges)

    if len(edges) <= 2:
        return np.zeros_like(x, dtype=int)

    out = np.digitize(x, edges[1:-1], right=False)
    return out.astype(int)


def generate_causal_data(n_samples=5000, noise_scale=0.5, seed=42, 
                         normalize=True, discretize=False, bins=20):
    """
    Generate synthetic data with graph:
        X -> Y -> Z
        W -> Y

    Parameters
    ----------
    n_samples : int
        Number of samples.
    noise_scale : float
        Standard deviation of Gaussian noise.
    seed : int
        Random seed.
    discretize : bool
        If True, discretize variables into quantile bins for histogram-based CE.
    bins : int
        Number of discrete bins if discretize=True.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns X, W, Y, Z.
    """
    rng = np.random.default_rng(seed)

    # Independent root causes
    X = rng.normal(0, 1, n_samples)
    W = rng.normal(0, 1, n_samples)

    # Y depends on X and W
    Y = 0.9 * X + 0.8 * W + rng.normal(0, noise_scale, n_samples)

    # Z depends only on Y
    Z = 1.0 * Y + rng.normal(0, noise_scale, n_samples)

    data = pd.DataFrame({"X": X, "W": W, "Y": Y, "Z": Z})

    if normalize:
        data = data.apply(lambda col: (col - col.mean()) / col.std())

    if discretize:
        data = data.apply(lambda col: discretize_quantiles(col.to_numpy(), bins=bins))

    return data

In [55]:
def greedy_parents(
    data,
    target_name,
    candidate_names=None,
    initial_conditioning_names=None,
    nbins=8,
    base=2,
    threshold=0.01,
    verbose=True,
):
    """
    Greedy iterative parent-selection using causation entropy.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame containing all variables.
    target_name : str
        Name of target variable.
    candidate_names : list[str] or None
        Candidate parent variables. If None, all other columns are used.
    initial_conditioning_names : list[str] or None
        Variables always included in the conditioning set at the start.
    nbins : int
        Number of bins for entropy estimation.
    base : float
        Logarithm base.
    threshold : float
        Stop when the best remaining CE score falls below this threshold.
    verbose : bool
        If True, print intermediate scores.

    Returns
    -------
    selected : list[str]
        Selected parent variables.
    history : list[dict]
        Per-iteration score summaries.
    """
    if candidate_names is None:
        candidate_names = [c for c in data.columns if c != target_name]

    if initial_conditioning_names is None:
        initial_conditioning_names = []

    target = data[target_name].to_numpy()
    conditioning_names = list(initial_conditioning_names)
    selected = []
    remaining = [c for c in candidate_names if c not in conditioning_names]

    history = []

    while remaining:
        conditioning_arrays = [data[name].to_numpy() for name in conditioning_names]
        scores = {}

        for name in remaining:
            source = data[name].to_numpy()
            score = causation_entropy(
                target,
                source,
                conditioning_set=tuple(conditioning_arrays),
                nbins=nbins,
                base=base,
            )
            scores[name] = score

        best_name = max(scores, key=scores.get)
        best_score = scores[best_name]

        step_info = {
            "target": target_name,
            "conditioning": conditioning_names.copy(),
            "scores": scores.copy(),
            "best_candidate": best_name,
            "best_score": best_score,
        }
        history.append(step_info)

        if verbose:
            print(f"\nTarget: {target_name}")
            print(f"Current conditioning set: {conditioning_names}")
            print("Candidate scores:")
            for k, v in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
                print(f"  {k}: {v:.6f}")
            print(f"Best candidate: {best_name}  |  score = {best_score:.6f}")

        if best_score < threshold:
            if verbose:
                print(f"Stopping because best score < threshold ({threshold}).")
            break

        selected.append(best_name)
        conditioning_names.append(best_name)
        remaining.remove(best_name)

    return selected, history

def leave_one_out_parents(
    data,
    target_name,
    candidate_names=None,
    nbins=8,
    base=2,
    threshold=0.2,
    verbose=True,
):
    """
    Identify parent candidates by comparing each candidate's marginal
    contribution against the full conditioning set.

    For each candidate c, compute

        ratio_c = [ H(T | all_except_c) - H(T | all_candidates) ] / H(T)

    and declare c a parent if ratio_c > threshold.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame containing all variables.
    target_name : str
        Name of target variable T.
    candidate_names : list[str] or None
        Candidate parent variables. If None, all columns except target_name are used.
    nbins : int
        Number of bins for histogram-based entropy estimation.
    base : float
        Logarithm base for entropy.
    threshold : float
        Normalized contribution threshold for declaring a parent.
    verbose : bool
        If True, print intermediate values.

    Returns
    -------
    parents : list[str]
        Candidates whose normalized contribution exceeds threshold.
    results : pandas.DataFrame
        Table with entropy values, raw contribution, normalized ratio,
        and parent decision for each candidate.
    """
    if candidate_names is None:
        candidate_names = [c for c in data.columns if c != target_name]

    target = data[target_name].to_numpy()

    # Entropy of the target
    H_target = entropy(target, nbins=nbins, base=base)
    if H_target <= 0:
        raise ValueError(f"Entropy of target '{target_name}' is zero, cannot normalize.")

    # Full conditioning set: all candidates together
    full_conditioning_arrays = [data[name].to_numpy() for name in candidate_names]

    if len(full_conditioning_arrays) == 0:
        raise ValueError("No candidate variables were provided.")

    H_target_given_all = conditional_entropy(
        target,
        *full_conditioning_arrays,
        nbins=nbins,
        base=base,
    )

    rows = []
    parents = []

    for candidate in candidate_names:
        other_names = [name for name in candidate_names if name != candidate]
        other_arrays = [data[name].to_numpy() for name in other_names]

        if len(other_arrays) == 0:
            H_target_given_others = H_target
        else:
            H_target_given_others = conditional_entropy(
                target,
                *other_arrays,
                nbins=nbins,
                base=base,
            )

        contribution = H_target_given_others - H_target_given_all
        ratio = contribution / H_target
        is_parent = ratio > threshold

        if is_parent:
            parents.append(candidate)

        rows.append(
            {
                "target": target_name,
                "candidate": candidate,
                "H_target": H_target,
                "H_target_given_others": H_target_given_others,
                "H_target_given_all": H_target_given_all,
                "contribution": contribution,
                "ratio": ratio,
                "is_parent": is_parent,
            }
        )

        if verbose:
            print(f"\nTarget: {target_name}")
            print(f"Candidate: {candidate}")
            print(f"H(T)                    = {H_target:.6f}")
            print(f"H(T | others)           = {H_target_given_others:.6f}")
            print(f"H(T | all)              = {H_target_given_all:.6f}")
            print(f"Contribution            = {contribution:.6f}")
            print(f"Contribution / H(T)     = {ratio:.6f}")
            print(f"Parent?                 = {is_parent}")

    results = pd.DataFrame(rows).sort_values("ratio", ascending=False).reset_index(drop=True)
    return parents, results

def adjusted_mutual_information_parents(
    data,
    target_name,
    candidate_names=None,
    nbins=8,
    base=2,
    threshold=0.01,
    verbose=True,
):
    """
    Identify parent candidates of a target using the score

        score(candidate) =
            [ MI(target; candidate) - H(target | others) ] / H(target)

    where:
        - MI(target; candidate) is the mutual information between the
          target and the candidate
        - H(target | others) is the conditional entropy of the target
          given all other candidates except the current candidate
        - H(target) is the entropy of the target

    A candidate is declared a parent if score(candidate) > threshold.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame containing target and candidate variables.
    target_name : str
        Name of the target variable.
    candidate_names : list[str] or None
        Candidate parent variables. If None, all columns except the target
        are used.
    nbins : int
        Number of bins for histogram-based entropy estimation.
    base : float
        Logarithm base used in entropy calculations.
    threshold : float
        Threshold for declaring a candidate a parent.
    verbose : bool
        If True, print intermediate results.

    Returns
    -------
    parents : list[str]
        Candidates whose score exceeds the threshold.
    results : pandas.DataFrame
        Detailed table with entropy, MI, conditional entropy, score,
        and parent decision for each candidate.
    """
    if candidate_names is None:
        candidate_names = [c for c in data.columns if c != target_name]

    if len(candidate_names) == 0:
        raise ValueError("No candidate variables were provided.")

    target = data[target_name].to_numpy()
    H_target = entropy(target, nbins=nbins, base=base)

    if H_target <= 0:
        raise ValueError(f"Entropy of target '{target_name}' is zero, cannot normalize.")

    # Full conditioning set: all candidates together
    full_conditioning_arrays = [data[name].to_numpy() for name in candidate_names]

    rows = []
    parents = []

    H_target_given_all = conditional_entropy(
        target,
        *full_conditioning_arrays,
        nbins=nbins,
        base=base,
    )

    for candidate in candidate_names:
        candidate_array = data[candidate].to_numpy()
        other_names = [name for name in candidate_names if name != candidate]
        other_arrays = [data[name].to_numpy() for name in other_names]

        mi_tc = mutual_information(
            target,
            candidate_array,
            nbins=nbins,
            base=base,
        )

        if len(other_arrays) == 0:
            H_target_given_others = H_target
        else:
            H_target_given_others = conditional_entropy(
                target,
                *other_arrays,
                nbins=nbins,
                base=base,
            )

        score = (mi_tc - H_target_given_others + H_target_given_all) / H_target
        is_parent = score > threshold

        if is_parent:
            parents.append(candidate)

        rows.append(
            {
                "target": target_name,
                "candidate": candidate,
                "H_target": H_target,
                "MI_target_candidate": mi_tc,
                "H_target_given_others": H_target_given_others,
                "score": score,
                "is_parent": is_parent,
            }
        )

        if verbose:
            print(f"\nTarget: {target_name}")
            print(f"Candidate: {candidate}")
            print(f"H(T)                 = {H_target:.6f}")
            print(f"I(T; candidate)      = {mi_tc:.6f}")
            print(f"H(T | others)        = {H_target_given_others:.6f}")
            print(f"H(T | all)           = {H_target_given_all:.6f}")
            print(f"Score                = {score:.6f}")
            print(f"Parent?              = {is_parent}")

    results = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    return parents, results


def infer_dependency_tree(data, nbins=8, base=2, threshold=0.01, verbose=True):
    """
    Infer a parent set for each variable using greedy causation entropy.

    Returns
    -------
    dict
        Mapping target -> list of inferred parents.
    """
    tree = {}
    for target in data.columns:
        candidates = [c for c in data.columns if c != target]
        parents, _ = adjusted_mutual_information_parents(
            data=data,
            target_name=target,
            candidate_names=candidates,
            nbins=nbins,
            base=base,
            threshold=threshold,
            verbose=verbose,
        )
        tree[target] = parents
    return tree

In [56]:
def plot_variables(data, columns=("X", "Y", "Z", "W"), max_points=500):
    """
    Visualize multiple variables from a DataFrame in a single figure.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame containing the variables.
    columns : tuple[str]
        Columns to plot.
    max_points : int
        Maximum number of samples to display (useful if dataset is large).
    """

    # Limit number of points for readability
    subset = data[list(columns)].iloc[:max_points]

    fig, axes = plt.subplots(len(columns), 1, figsize=(10, 6), sharex=True)

    for i, col in enumerate(columns):
        axes[i].plot(subset.index, subset[col], linewidth=1)
        axes[i].set_ylabel(col)
        axes[i].grid(alpha=0.3)

    axes[-1].set_xlabel("Sample index")
    fig.suptitle("Generated Variables: X, Y, Z, W")

    plt.tight_layout()
    plt.show()

import matplotlib.pyplot as plt


def plot_histograms(data, columns=("X", "Y", "Z", "W"), bins=30):
    """
    Plot histograms of variables in one figure.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame containing variables.
    columns : tuple[str]
        Column names to plot.
    bins : int
        Number of histogram bins.
    """

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))

    axes = axes.flatten()

    for i, col in enumerate(columns):
        axes[i].hist(data[col], bins=bins)
        axes[i].set_title(f"Distribution of {col}")
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()

In [58]:
data = generate_causal_data(
    n_samples=500,
    noise_scale=0.4,
    seed=7,
    normalize=False,
    discretize=False,
    bins=20,
)

# plot_variables(data, data.columns)
# plot_histograms(data, data.columns)

print("First few rows of generated data:")
print(data.head())

print("\nEntropy of each distribution:")
cols = list(data.columns)
for i, a in enumerate(cols):
    en = entropy(data[a].to_numpy(), nbins=20)
    print(f"E({a}) = {en:.6f}")

print("\nPairwise mutual information:")
cols = list(data.columns)
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        mi = mutual_information(data[a].to_numpy(), data[b].to_numpy(), nbins=20)
        print(f"I({a}; {b}) = {mi:.6f}")

print("\n" + "=" * 30)
print("Inferring dependency tree")
print("=" * 30)

tree = infer_dependency_tree(
    data,
    nbins=20,
    threshold=0.1,
    verbose=True,
)

print("\nRecovered dependency tree:")
for target, parents in tree.items():
    print(f"{parents} -> {target}")

First few rows of generated data:
          X         W         Y         Z
0  0.001230 -0.285416 -0.083704 -0.571014
1  0.298746 -1.506279 -0.779515 -0.121685
2 -0.274138 -0.977258 -1.196803 -0.994008
3 -0.890592  1.385839  1.115494  1.831044
4 -0.454671  0.820579  0.395675  0.714430

Entropy of each distribution:
E(X) = 3.711153
E(W) = 3.842368
E(Y) = 3.681282
E(Z) = 3.625298

Pairwise mutual information:
I(X; W) = 0.482757
I(X; Y) = 0.770520
I(X; Z) = 0.688658
I(W; Y) = 0.681986
I(W; Z) = 0.656824
I(Y; Z) = 1.612204

Inferring dependency tree

Target: X
Candidate: W
H(T)                 = 3.711153
I(T; candidate)      = 0.482757
H(T | others)        = 2.301304
H(T | all)           = 0.615634
Score                = -0.324135
Parent?              = False

Target: X
Candidate: Y
H(T)                 = 3.711153
I(T; candidate)      = 0.770520
H(T | others)        = 1.499090
H(T | all)           = 0.615634
Score                = -0.030432
Parent?              = False

Target: X
Candidate